# Training Linear Regression and XGBoost models on data

## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")

### Load Data

In [2]:
train_transformed = pd.read_csv('../../data/preprocessed/train_preprocessed.csv')
x_train_transformed = train_transformed.drop("diabetes", axis=1)
y_train_transformed = train_transformed["diabetes"]

val = pd.read_csv('../../data/preprocessed/validation_preprocessed.csv')
x_val = val.drop("diabetes", axis=1)
y_val = val["diabetes"]

print(f'x_train_transformed: {train_transformed.shape}')
train_transformed.head()

x_train_transformed: (67139, 21)


,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,glucose_hba1c_interaction,...,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag,diabetes
0,0.0,0.0,0.0,0.0,0.0,0.612112,-0.637209,0.000000,-0.677966,-0.459103,...,-0.056612,-0.232854,-0.081827,0,0,0,0,0,0,0
1,0.0,0.0,1.0,0.0,0.0,0.737237,0.818605,0.500000,0.000000,0.411609,...,0.658534,1.018393,0.557564,0,0,0,0,0,0,0
2,0.0,0.0,1.0,0.0,0.0,0.399399,-0.229457,0.000000,0.254237,0.382586,...,-0.339001,0.014118,-0.070409,1,0,0,0,0,0,0
3,0.0,1.0,0.0,0.0,0.0,0.874875,1.485271,0.500000,0.254237,0.668865,...,1.258590,1.470922,1.050428,0,0,0,0,1,0,0
4,0.0,0.0,0.0,1.0,0.0,0.687187,-0.395349,-1.642857,1.016949,-0.142480,...,0.148131,-1.008759,1.078972,1,0,0,0,0,0,0


In [3]:
train_ADASYN = pd.read_csv('../../data/imbalance_resolve/ADASYN.csv')
x_train_ADASYN = train_ADASYN.drop("diabetes_target", axis=1)
y_train_ADASYN = train_ADASYN["diabetes_target"]
print(f'train_ADASYN: {train_ADASYN.shape}')

train_smote_enn = pd.read_csv('../../data/imbalance_resolve/train_smote_enn.csv')
x_train_smote_enn = train_smote_enn.drop("diabetes_target", axis=1)
y_train_smote_enn = train_smote_enn["diabetes_target"]
print(f'train_smote_enn: {train_smote_enn.shape}')

train_smote_tomek = pd.read_csv('../../data/imbalance_resolve/train_smote_tomek.csv')
x_train_smote_tomek= train_smote_tomek.drop("diabetes_target", axis=1)
y_train_smote_tomek = train_smote_tomek["diabetes_target"]
print(f'train_smote_tomek: {train_smote_tomek.shape}')

train_smote = pd.read_csv('../../data/imbalance_resolve/train_smote.csv')
x_train_smote = train_smote.drop("diabetes_target", axis=1)
y_train_smote = train_smote["diabetes_target"]
print(f'train_smote: {train_smote.shape}')

train_ADASYN: (122666, 16)
train_smote_enn: (112303, 16)
train_smote_tomek: (121658, 16)
train_smote: (112303, 16)


### Evaluation Metrics
- Accuracy
- **Precision**: When I say diabetic, how often am I right?, TP / (TP + FP)
- **Recall**: How many diabetics did I catch?, TP / (TP + FN) :: **most important**
- **F1-score**
- ROC-AUC
- **PR-AUC**
- **MCC**

`it is more important to us to say that a patient has diabetes when he has, and missing that would be bad that is why recall is most important here`

In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,  # PR-AUC
    matthews_corrcoef
)

#### Helper function for model evaluation

In [5]:
def evaluate_model(model, X, y, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]

    y_pred = (y_proba >= threshold).astype(int)

    results = {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred, zero_division=0),
        "Recall": recall_score(y, y_pred),
        "F1-score": f1_score(y, y_pred),
        "ROC-AUC": roc_auc_score(y, y_proba),
        "PR-AUC": average_precision_score(y, y_proba),
        "MCC": matthews_corrcoef(y, y_pred)
    }    

    return results

### Logistic Regression

I will first try logistic regression using different train data and select the best data to tune the hyperparameters:
- x_train_transformed
- x_train_transformed (with weights)
- x_train_ADASYN
- x_train_smote_enn
- x_train_smote_tomek
- x_train_smote

#### import model

In [6]:
from sklearn.linear_model import LogisticRegression

#### x_train_transformed

In [7]:
model_LR_transformed = LogisticRegression(max_iter=1000, random_state=42)

model_LR_transformed.fit(x_train_transformed, y_train_transformed)

results = evaluate_model(model_LR_transformed, x_val, y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9607
Precision: 0.8716
Recall: 0.6509
F1-score: 0.7453
ROC-AUC: 0.9622
PR-AUC: 0.8298
MCC: 0.7335


#### x_train_transformed (with weights)

In [8]:
model_LR_transformed_w = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")

model_LR_transformed_w.fit(x_train_transformed, y_train_transformed)

results = evaluate_model(model_LR_transformed_w, x_val, y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.8864
Precision: 0.4308
Recall: 0.8852
F1-score: 0.5795
ROC-AUC: 0.9628
PR-AUC: 0.8251
MCC: 0.5682


#### x_train_ADASYN

In [9]:
model_LR_ADASYN = LogisticRegression(max_iter=1000, random_state=42)

model_LR_ADASYN.fit(x_train_ADASYN, y_train_ADASYN)

results = evaluate_model(model_LR_ADASYN, x_val[x_train_ADASYN.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.7990
Precision: 0.2996
Recall: 0.9520
F1-score: 0.4558
ROC-AUC: 0.9586
PR-AUC: 0.8035
MCC: 0.4650


#### x_train_smote_enn

In [10]:
model_LR_smote_enn = LogisticRegression(max_iter=1000, random_state=42)

model_LR_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_LR_smote_enn, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.8508
Precision: 0.3647
Recall: 0.9269
F1-score: 0.5234
ROC-AUC: 0.9602
PR-AUC: 0.8098
MCC: 0.5239


#### x_train_smote_tomek

In [11]:
model_LR_smote_tomek = LogisticRegression(max_iter=1000, random_state=42)

model_LR_smote_tomek.fit(x_train_smote_tomek, y_train_smote_tomek)

results = evaluate_model(model_LR_smote_tomek, x_val[x_train_smote_tomek.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.8649
Precision: 0.3870
Recall: 0.9033
F1-score: 0.5419
ROC-AUC: 0.9594
PR-AUC: 0.8079
MCC: 0.5363


#### x_train_smote

In [12]:
model_LR_smote = LogisticRegression(max_iter=1000, random_state=42)

model_LR_smote.fit(x_train_smote, y_train_smote)

results = evaluate_model(model_LR_smote, x_val[x_train_smote.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.8508
Precision: 0.3647
Recall: 0.9269
F1-score: 0.5234
ROC-AUC: 0.9602
PR-AUC: 0.8098
MCC: 0.5239


#### Results

| Method                     | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC   |
|---------------------------|----------|-----------|--------|----------|---------|--------|-------|
| x_train_transformed       | `0.9607`   | `0.8716`    | 0.6509 | `0.7453`   | 0.9622  | `0.8298` | `0.7335` |
| x_train_transformed (w)   | 0.8864   | 0.4308    | 0.8852 | 0.5795   | `0.9628`  | 0.8251 | 0.5682 |
| x_train_ADASYN            | 0.7990   | 0.2996    | `0.9520` | 0.4558   | 0.9586  | 0.8035 | 0.4650 |
| x_train_smote_enn         | 0.8508   | 0.3647    | 0.9269 | 0.5234   | 0.9602  | 0.8098 | 0.5239 |
| x_train_smote_tomek       | 0.8649   | 0.3870    | 0.9033 | 0.5419   | 0.9594  | 0.8079 | 0.5363 |
| x_train_smote             | 0.8508   | 0.3647    | 0.9269 | 0.5234   | 0.9602  | 0.8098 | 0.5239 |

we will continue with the x_train_smote_enn to tune the hyperparemeters

#### Tuning hyperparameters using x_train_smote_enn

##### E1

In [13]:
model_LR_smote_enn = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

model_LR_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_LR_smote_enn, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

c:\Users\ahmed\AppData\Local\pypoetry\Cache\virtualenvs\diabetes-prediction-7fFvrh6q-py3.13\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Accuracy: 0.8536
Precision: 0.3693
Recall: 0.9261
F1-score: 0.5280
ROC-AUC: 0.9603
PR-AUC: 0.8100
MCC: 0.5281


##### E2

In [14]:
model_LR_smote_enn = LogisticRegression(
    penalty='l2',
    C=0.1,
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

model_LR_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_LR_smote_enn, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

c:\Users\ahmed\AppData\Local\pypoetry\Cache\virtualenvs\diabetes-prediction-7fFvrh6q-py3.13\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Accuracy: 0.8553
Precision: 0.3718
Recall: 0.9237
F1-score: 0.5302
ROC-AUC: 0.9605
PR-AUC: 0.8109
MCC: 0.5296


##### E3

In [15]:
model_LR_smote_enn = LogisticRegression(
    penalty='l2',
    C=0.01,
    solver='liblinear',
    class_weight='balanced',    
    max_iter=1000,
    random_state=42
)

model_LR_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_LR_smote_enn, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

c:\Users\ahmed\AppData\Local\pypoetry\Cache\virtualenvs\diabetes-prediction-7fFvrh6q-py3.13\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Accuracy: 0.8594
Precision: 0.3786
Recall: 0.9206
F1-score: 0.5365
ROC-AUC: 0.9611
PR-AUC: 0.8142
MCC: 0.5349


##### E4

In [16]:
model_LR_smote_enn = LogisticRegression(
    penalty='l2',
    C=0.001,
    solver='liblinear',
    class_weight='balanced',    
    max_iter=1000,
    random_state=42
)

model_LR_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_LR_smote_enn, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.8546
Precision: 0.3703
Recall: 0.9206
F1-score: 0.5282
ROC-AUC: 0.9605
PR-AUC: 0.8143
MCC: 0.5271


c:\Users\ahmed\AppData\Local\pypoetry\Cache\virtualenvs\diabetes-prediction-7fFvrh6q-py3.13\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


#### Results

| C value | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC   |
|--------|----------|-----------|--------|----------|---------|--------|-------|
| C=1.0  | 0.8536   | 0.3693    | 0.9261 | 0.5280   | 0.9603  | 0.8100 | 0.5281 |
| C=0.1  | 0.8553   | 0.3718    | 0.9237 | 0.5302   | 0.9605  | 0.8109 | 0.5296 |
| C=0.01 | 0.8631   | 0.3849    | 0.9167 | 0.5422   | 0.9612  | 0.8149 | 0.5394 |
| C=0.001 | 0.8546   | 0.3703    | 0.9206 | 0.5282   | 0.9605  | 0.8143 | 0.5271 |


`choose C=0.01`: it gives good trade off between Accuracy, Precision, Recall

#### Parameters for Logistic Regression

| Parameter       | Value        | Description |
|----------------|-------------|-------------|
| penalty        | l2          | L2 regularization (ridge) to reduce overfitting |
| C              | 0.01        | Inverse of regularization strength (smaller = stronger regularization) |
| solver         | liblinear   | Optimization algorithm suitable for small datasets and L1/L2 penalties |
| class_weight   | balanced    | Automatically adjusts weights inversely proportional to class frequencies |
| max_iter       | 1000        | Maximum number of optimization iterations |
| random_state   | 42          | Seed for reproducibility |

### XGBoost

In [17]:
from xgboost import XGBClassifier

#### x_train_transformed (with weights)

In [18]:
neg = (y_train_transformed == 0).sum()
pos = (y_train_transformed == 1).sum()

scale_pos_weight = neg / pos

model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.05, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.8,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 scale_pos_weight=scale_pos_weight, # Balances positive/negative classes. #>1 helps with class imbalance (minority class).
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )


model_XGB_transformed_w.fit(x_train_transformed, y_train_transformed)

results = evaluate_model(model_XGB_transformed_w, x_val, y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9169
Precision: 0.5174
Recall: 0.9002
F1-score: 0.6571
ROC-AUC: 0.9773
PR-AUC: 0.8861
MCC: 0.6443


#### x_train_smote_enn

In [19]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.05, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.8,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9317
Precision: 0.5744
Recall: 0.8766
F1-score: 0.6941
ROC-AUC: 0.9779
PR-AUC: 0.8898
MCC: 0.6761


#### Results

| Method                         | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC   |
|--------------------------------|----------|-----------|--------|----------|---------|--------|-------|
| x_train_transformed (weights)  | 0.9169   | 0.5174    | `0.9002` | 0.6571   | 0.9773  | 0.8861 | 0.6443 |
| x_train_smote_enn              | `0.9317`   | `0.5744`    | 0.8766 | `0.6941`   | `0.9779`  | `0.8898` | `0.6761` |

`will continue with the x_train_smote_enn as it gives overall better metrics`

##### E1

In [20]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.05, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.8,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9317
Precision: 0.5744
Recall: 0.8766
F1-score: 0.6941
ROC-AUC: 0.9779
PR-AUC: 0.8898
MCC: 0.6761


##### E2
try `n_estimators=200`

In [21]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=200, # Number of boosting rounds (trees). 
 learning_rate=0.05, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.8,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9251
Precision: 0.5472
Recall: 0.8836
F1-score: 0.6759
ROC-AUC: 0.9774
PR-AUC: 0.8878
MCC: 0.6595


##### E3
continue with `n_estimators = 300`
different learning rates

In [22]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.05, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.8,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9317
Precision: 0.5744
Recall: 0.8766
F1-score: 0.6941
ROC-AUC: 0.9779
PR-AUC: 0.8898
MCC: 0.6761


In [23]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.01, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.8,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.8816
Precision: 0.4227
Recall: 0.9285
F1-score: 0.5809
ROC-AUC: 0.9750
PR-AUC: 0.8788
MCC: 0.5781


In [24]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.1, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.8,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9388
Precision: 0.6078
Recall: 0.8687
F1-score: 0.7152
ROC-AUC: 0.9782
PR-AUC: 0.8901
MCC: 0.6958


In [25]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.8,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9450
Precision: 0.6375
Recall: 0.8750
F1-score: 0.7376
ROC-AUC: 0.9785
PR-AUC: 0.8929
MCC: 0.7188


##### E4
continue with `learning_rate=0.25`

try different `max_depth`

In [26]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.8,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9450
Precision: 0.6375
Recall: 0.8750
F1-score: 0.7376
ROC-AUC: 0.9785
PR-AUC: 0.8929
MCC: 0.7188


In [27]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=5, # Maximum depth of each tree. 
 subsample=0.8,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9418
Precision: 0.6230
Recall: 0.8640
F1-score: 0.7240
ROC-AUC: 0.9779
PR-AUC: 0.8906
MCC: 0.7039


In [28]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=7, # Maximum depth of each tree. 
 subsample=0.8,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9472
Precision: 0.6506
Recall: 0.8711
F1-score: 0.7449
ROC-AUC: 0.9789
PR-AUC: 0.8938
MCC: 0.7256


##### E5
continue with `max_depth=6`

try different `subsample`

In [29]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.8,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9450
Precision: 0.6375
Recall: 0.8750
F1-score: 0.7376
ROC-AUC: 0.9785
PR-AUC: 0.8929
MCC: 0.7188


In [30]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9452
Precision: 0.6394
Recall: 0.8726
F1-score: 0.7380
ROC-AUC: 0.9784
PR-AUC: 0.8932
MCC: 0.7189


In [31]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.6,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9443
Precision: 0.6343
Recall: 0.8742
F1-score: 0.7352
ROC-AUC: 0.9781
PR-AUC: 0.8909
MCC: 0.7163


In [32]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.4,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9447
Precision: 0.6362
Recall: 0.8758
F1-score: 0.7370
ROC-AUC: 0.9781
PR-AUC: 0.8923
MCC: 0.7183


In [33]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.9,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9455
Precision: 0.6420
Recall: 0.8671
F1-score: 0.7378
ROC-AUC: 0.9785
PR-AUC: 0.8926
MCC: 0.7181


##### E6
will continue with `subsample=0.7`

try different `colsample_bytree`

In [34]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9452
Precision: 0.6394
Recall: 0.8726
F1-score: 0.7380
ROC-AUC: 0.9784
PR-AUC: 0.8932
MCC: 0.7189


In [35]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.7, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9447
Precision: 0.6368
Recall: 0.8711
F1-score: 0.7357
ROC-AUC: 0.9786
PR-AUC: 0.8926
MCC: 0.7165


In [36]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.9, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss'  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9445
Precision: 0.6358
Recall: 0.8703
F1-score: 0.7348
ROC-AUC: 0.9782
PR-AUC: 0.8926
MCC: 0.7155


##### E7

will continue with `colsample_bytree=0.8`

try different `min_child_weight(1, 3, 5)`

In [37]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss' # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9452
Precision: 0.6394
Recall: 0.8726
F1-score: 0.7380
ROC-AUC: 0.9784
PR-AUC: 0.8932
MCC: 0.7189


In [38]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss',  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 min_child_weight=1
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9452
Precision: 0.6394
Recall: 0.8726
F1-score: 0.7380
ROC-AUC: 0.9784
PR-AUC: 0.8932
MCC: 0.7189


In [39]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss',  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 min_child_weight=3
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9445
Precision: 0.6346
Recall: 0.8766
F1-score: 0.7362
ROC-AUC: 0.9781
PR-AUC: 0.8921
MCC: 0.7176


##### E8

will continue with `min_child_weight=1`

try different `gamma(0, 0.1, 0.3, 0.5, 1)`

In [40]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss',  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 min_child_weight=1, 
 gamma=0
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9452
Precision: 0.6394
Recall: 0.8726
F1-score: 0.7380
ROC-AUC: 0.9784
PR-AUC: 0.8932
MCC: 0.7189


In [41]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss',  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 min_child_weight=1, 
 gamma=0.1
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9444
Precision: 0.6339
Recall: 0.8781
F1-score: 0.7363
ROC-AUC: 0.9781
PR-AUC: 0.8918
MCC: 0.7179


In [42]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss',  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 min_child_weight=1, 
 gamma=0.3
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9453
Precision: 0.6394
Recall: 0.8742
F1-score: 0.7386
ROC-AUC: 0.9783
PR-AUC: 0.8920
MCC: 0.7197


In [43]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss',  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 min_child_weight=1, 
 gamma=0.5
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9461
Precision: 0.6444
Recall: 0.8719
F1-score: 0.7411
ROC-AUC: 0.9787
PR-AUC: 0.8934
MCC: 0.7219


In [44]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss',  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 min_child_weight=1, 
 gamma=1
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9425
Precision: 0.6262
Recall: 0.8679
F1-score: 0.7275
ROC-AUC: 0.9781
PR-AUC: 0.8910
MCC: 0.7079


`choose gamma = 0.5`

#### Parameters for XGBoost

| Parameter            | Value  | Description |
|---------------------|--------|-------------|
| n_estimators        | 300    | Number of boosting rounds (trees) |
| learning_rate       | 0.25   | Step size shrinkage; higher = faster learning, lower generalization risk tuning |
| max_depth           | 6      | Maximum depth of each tree |
| subsample           | 0.7    | Fraction of samples used per tree |
| colsample_bytree    | 0.8    | Fraction of features used per tree |
| random_state        | 42     | Seed for reproducibility |
| eval_metric         | logloss| Loss function used for evaluation (logistic loss) |
| min_child_weight    | 1      | Minimum sum of instance weight in a child node |
| gamma               | 0.5    | Minimum loss reduction required to make a split |

### Evaluating on test set

In [51]:
import sys
sys.path.append(r"C:\Users\ahmed\OneDrive\Desktop\Semester work\Data Science\Project\Diabetes-Prediction")

from src.diabetes_prediction.transformation.transformation import DataTransformation

transformer = DataTransformation()
transformer.load_preprocessor('../Transformation/preprocessor.pkl')

test = pd.read_csv('../../data/split/test.csv')
x_test = test.drop("diabetes", axis=1)
y_test = test["diabetes"]

x_test_transformed = transformer.transform(x_test)
x_test_transformed = x_test_transformed[x_train_smote_enn.columns]

#### Logistic Regression

In [53]:
model_LR_smote_enn_final = LogisticRegression(
    penalty='l2',
    C=0.01,
    solver='liblinear',
    class_weight='balanced',    
    max_iter=1000,
    random_state=42
)

model_LR_smote_enn_final.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_LR_smote_enn_final, x_test_transformed, y_test)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

c:\Users\ahmed\AppData\Local\pypoetry\Cache\virtualenvs\diabetes-prediction-7fFvrh6q-py3.13\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Accuracy: 0.8558
Precision: 0.3715
Recall: 0.9127
F1-score: 0.5281
ROC-AUC: 0.9575
PR-AUC: 0.7897
MCC: 0.5253


In [47]:
model_LR_smote_enn = LogisticRegression(
    penalty='l2',
    C=0.01,
    solver='liblinear',
    class_weight='balanced',    
    max_iter=1000,
    random_state=42
)

model_LR_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_LR_smote_enn, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

c:\Users\ahmed\AppData\Local\pypoetry\Cache\virtualenvs\diabetes-prediction-7fFvrh6q-py3.13\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Accuracy: 0.8594
Precision: 0.3786
Recall: 0.9206
F1-score: 0.5365
ROC-AUC: 0.9611
PR-AUC: 0.8142
MCC: 0.5349


#### XGBoost

In [54]:
model_XGB_transformed_final = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss',  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 min_child_weight=1, 
 gamma=0.5
 )

model_XGB_transformed_final.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_final, x_test_transformed, y_test)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9408
Precision: 0.6180
Recall: 0.8648
F1-score: 0.7208
ROC-AUC: 0.9780
PR-AUC: 0.8839
MCC: 0.7009


In [49]:
model_XGB_transformed_w = xgb_clf = XGBClassifier(
 n_estimators=300, # Number of boosting rounds (trees). 
 learning_rate=0.25, # Smaller values make learning slower but improve generalization.
 max_depth=6, # Maximum depth of each tree. 
 subsample=0.7,  # Fraction of training samples used per tree. 
 colsample_bytree=0.8, # Fraction of features used per tree. 
 random_state=42,  # Random seed for reproducibility. 
 eval_metric='logloss',  # Evaluation metric. 'logloss' = logistic loss (cross-entropy)   )
 min_child_weight=1, 
 gamma=0.5
 )

model_XGB_transformed_w.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_XGB_transformed_w, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9461
Precision: 0.6444
Recall: 0.8719
F1-score: 0.7411
ROC-AUC: 0.9787
PR-AUC: 0.8934
MCC: 0.7219


## Final Results

| Model               | Split | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC   |
|--------------------|-------|----------|-----------|--------|----------|---------|--------|-------|
| Logistic Regression | Test  | 0.8558   | 0.3715    | 0.9127 | 0.5281   | 0.9575  | 0.7897 | 0.5253 |
| Logistic Regression | Val   | 0.8594   | 0.3786    | 0.9206 | 0.5365   | 0.9611  | 0.8142 | 0.5349 |
| XGBoost            | Test  | 0.9408   | 0.6180    | 0.8648 | 0.7208   | 0.9780  | 0.8839 | 0.7009 |
| XGBoost            | Val   | 0.9461   | 0.6444    | 0.8719 | 0.7411   | 0.9787  | 0.8934 | 0.7219 |

`XGBoost` is more prefereable

### Save the models

In [55]:
import joblib

joblib.dump(model_LR_smote_enn_final, "../../models/LR_model.pkl")
joblib.dump(model_XGB_transformed_final, "../../models/xgb_model.pkl")

['../../models/xgb_model.pkl']